# Transformer-based Image-captioning Model

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"kanayevarma","key":"b63fa144d0213119c56d8ff25d46c584"}'}

In [ ]:
%mkdir ~/.kaggle
%cp kaggle.json ~/.kaggle/

In [ ]:
%pip install kaggle

In [ ]:
!kaggle datasets -h

usage: kaggle datasets [-h]
                       {list,files,download,create,version,init,metadata,status,delete}
                       ...

options:
  -h, --help            show this help message and exit

commands:
  {list,files,download,create,version,init,metadata,status,delete}
    delete              Delete a dataset
    list                List available datasets
    files               List dataset files
    download            Download dataset files
    create              Create a new dataset
    version             Create a new dataset version
    init                Initialize metadata file for dataset creation
    metadata            Download metadata about a dataset
    status              Get the creation status for a dataset


In [ ]:
!pip install opendatasets
import opendatasets as od
od.download('https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset')

Dataset URL: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset


100%|██████████| 25.0G/25.0G [29:15<00:00, 15.3MB/s]


These are the modules we will need.

In [ ]:
%pip install tensordict
%pip install gensim

In [ ]:
import torch, numpy as np
from torch import nn, optim, tensor, Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights
from torchvision.transforms import v2
import tensordict as td

import nltk, gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors

import json
from PIL import Image
from random import shuffle
import shutil

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

We need to define `BASE_PATH` as the path where the data is stored, and `SAVE_TO_PATH` as our working directory where created files can be stored.

In [ ]:
BASE_PATH = '/content/coco-2017-dataset/coco2017'
SAVE_TO_PATH = '/content'

## Step 1: Pre-processing

We are going to start processing the data. First, let's create the `Vocab` class. This will be the basis for the vocabulary our model will understand and use to create embeddings. For our task, we will initialise our embeddings using pre-trained GloVe vectors.

In [ ]:
class Vocab:
    def __init__(self, tokens : set[str]):
        self.glove : KeyedVectors = api.load('glove-twitter-25') # type:ignore

        # limit vocabulary
        self.vocab_size = len(tokens) + 4
        self.embedding = torch.randn(self.vocab_size, 25)
        self.embedding[0] = torch.from_numpy(self.glove.get_vector('unk').copy())
        self.word2idx = {
            'unk': 0,
            '<start>' : 1,
            '<stop>' : 2,
            '<pad>' : 3
        }
        self.idx2word = ['unk', '<start>', '<stop>', '<pad>']

        valid_tokens = set()
        for token in tokens:
            if self.glove.has_index_for(token):
                self.word2idx[token] = len(self.idx2word)
                self.idx2word.append(token)
                self.embedding[self.word2idx[token]] = torch.from_numpy(self.glove.get_vector(token).copy())
                valid_tokens.add(token)

        self.tokens = valid_tokens
        self.vocab_size = len(valid_tokens) + 4
        self.embedding = self.embedding[:self.vocab_size]

    @staticmethod
    def from_file(base_path):
        tokens : set[str] = set()
        with open(base_path + '/annotations/captions_train2017.json') as f:
            data = json.load(f)
            for item in data['annotations']:
                caption : str = item['caption']
                for token in nltk.word_tokenize(caption.lower()): tokens.add(token)

        return Vocab(tokens)

In [ ]:
vocab = Vocab.from_file(BASE_PATH)

[==================================================] 100.0% 104.8/104.8MB downloaded


Our vocab is ready. The next step is to prepare for image-processing. All images will be passed through VGG19's featurisation layer before being passed into the transformer's encoder-decoder structure. To speed-up pre-processing, we will create the VGG model externally and pass all images through it before training the model.

In [ ]:
class VGGModel(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT)
        for param in vgg.parameters(): param.requires_grad_(False)
        layers = list(vgg.features.children())
        layers.pop()
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
vgg = VGGModel()
vgg.eval()
vgg.to('cuda')

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:06<00:00, 83.8MB/s]


VGGModel(
  (model): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), pa

Now it's time for data-preprocessing. Instead of processing the data one-at-a-time, we are going to speed it up as follows.
- The image captions can be handled one-at-a-time because tokenisation is relatively fast
- The images are processed by VGG in batches. This requires a `DataLoader`
- The resulting tensors are stored in a memory-mapped `TensorDict` for fast access later on.

In [ ]:
class Handler(Dataset):
    def __init__(self, images):
        '''input lists of paths'''
        self.images = images
        self.trans = v2.Compose([
            v2.Resize(256),
            v2.CenterCrop(224),
            v2.PILToTensor(),
            v2.ToDtype(dtype=torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        with Image.open(self.images[i]) as img:
            return self.trans(img)

In [ ]:
class CocoData:
    def __init__(self, base_path, save_to_path, upload_to_path, shard_size, num_shards, need_to_save, cat):

        self.dataset_paths = []
        f = open(base_path + f'/annotations/captions_{cat}2017.json')
        data = json.load(f)
        annotations = data['annotations']
        N_items = len(annotations)

        idx = 0
        cnt = 0

        while idx < N_items:
            file_path = save_to_path + f'/data/{cat}/{cnt}'

            if need_to_save:
                n = min(N_items - idx, shard_size)
                shard = self.get_empty_dict(n)
                images = []
                for i in range(n):
                    item = annotations[idx]

                    image_id = item['image_id']
                    caption = item['caption']

                    image_path = base_path + f'/{cat}2017/' + '%012d' % image_id + '.jpg'
                    images.append(image_path)

                    input_tensor, output_tensor = self.vectorise_caption(caption)

                    shard['input_captions'][i] = input_tensor
                    shard['output_captions'][i] = output_tensor

                    idx += 1
                handler = Handler(images)
                loader = DataLoader(handler, batch_size=64, num_workers=4, shuffle=False, pin_memory=True)
                i = 0
                with torch.no_grad():
                    for batch in loader:
                        batch_size = len(batch)
                        img_tensors = vgg(batch.to('cuda', non_blocking=True))
                        img_tensors = img_tensors.flatten(2).transpose(1, 2).cpu().to(torch.float16)
                        shard['images'][i:i+batch_size] = img_tensors
                        i += batch_size


                shard.memmap_(file_path)
                print('Finished memmap file', file_path)
                # To prevent repearing this step, the option to upload to Google Drive
                # is provided on Google Colab
                if upload_to_path is not None:
                    shutil.copytree(file_path, upload_to_path + file_path)

            self.dataset_paths.append(file_path)
            cnt += 1
            if len(self.dataset_paths) >= num_shards: break


    # helper functions
    def get_empty_dict(self, N):
        return td.TensorDict({
            'images': td.MemoryMappedTensor.empty((N, 196, 512), dtype=torch.float16),
            'input_captions': td.MemoryMappedTensor.empty((N, 20), dtype=torch.long),
            'output_captions': td.MemoryMappedTensor.empty((N, 20), dtype=torch.long)
        }, batch_size=[N])

    def vectorise_caption(self, caption : str):
        caption_tokens = nltk.word_tokenize(caption.lower())
        caption_vector = []
        for token in caption_tokens:
            if token in vocab.tokens:
                caption_vector.append(vocab.word2idx[token])
            else:
                caption_vector.append(0) # unk
            if len(caption_vector) == 19: break

        input_vector = [1] + caption_vector
        while len(input_vector) < 20: input_vector += [3]
        output_vector = caption_vector + [2]
        while len(output_vector) < 20: output_vector += [3]
        return tensor(input_vector, dtype=torch.long), tensor(output_vector, dtype=torch.long)

Now we can generate the training and validation datasets.

In [ ]:
train_data = CocoData(BASE_PATH, SAVE_TO_PATH, None, 20000, 10, True, 'train')
train_dataloaders = [
    DataLoader(
        td.load_memmap(path),
        batch_size=64,
        shuffle=True,
        num_workers=2,
        collate_fn=lambda x : x
    ) for path in train_data.dataset_paths
]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Finished memmap file /content/data/train/0
Finished memmap file /content/data/train/1
Finished memmap file /content/data/train/2
Finished memmap file /content/data/train/3
Finished memmap file /content/data/train/4
Finished memmap file /content/data/train/5
Finished memmap file /content/data/train/6
Finished memmap file /content/data/train/7
Finished memmap file /content/data/train/8
Finished memmap file /content/data/train/9


In [ ]:
val_data = CocoData(BASE_PATH, SAVE_TO_PATH, None, 20000, 1, True, 'val')
val_dataloaders = [
    DataLoader(
        td.load_memmap(path),
        batch_size=64,
        shuffle=False,
        num_workers=2,
        collate_fn=lambda x : x
    ) for path in val_data.dataset_paths
]

In [ ]:
%cd data

/content/data


In [ ]:
!kaggle datasets init

Data package template written to: /content/data/dataset-metadata.json


In [ ]:
!kaggle datasets create -p . --dir-mode zip

[Errno 28] No space left on device


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%rm -rf /content/coco-2017-dataset/

In [ ]:
%cd /content/data

/content/data


In [ ]:
!kaggle datasets create -p . --dir-mode zip

Starting upload for file train.zip
100% 5.54G/5.54G [06:12<00:00, 16.0MB/s]
Upload successful: train.zip (6GB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/kanayevarma/coco-tensors-memmap
